In [1]:
# from google.colab import drive

# drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !pip install -q scipy
# !pip install -q h5py
# !pip install -q matplotlib
# !pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
# !pip install -q transformers
# !pip install -q fuzzy_match
# !pip install -q nltk
# !pip install -q rouge
# !pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 39.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.4 MB/s eta 0:00:00


In [1]:
# %cd /content/drive/MyDrive/Research/FINAL/Code/MMMM

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [1]:
%cd /home/qid/MMMM

/home/qid/MMMM


In [12]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim

from master_init import *
from DSG import *

In [19]:
# config = {
#     "device" : "cuda:0",
#     "device_ids" : [0]
# }

config = {
    "device" : "cuda",
    "device_ids" : [0,1,2,3],
}
device = config["device"]
device_ids = config["device_ids"]

model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device, dtype=torch.float16)

In [20]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["ZuCo-BART"],
    bsz=[64]
)

In [21]:
def train(args_dict):
    dataloader = args_dict["dataloader"]
    model = args_dict["model"]
    optimizer = args_dict["optimizer"]
    tokenizer = args_dict["tokenizer"]
    criterion = args_dict["criterion"]
    device = args_dict["device"] if "device" in args_dict else "cuda"
    device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
    staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
    if staging_device==None:
        staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
    results = {}
    for phase in ['train', 'dev']:
        if phase == 'train':
            model.train()    # Set model to training mode
        else:
            model.eval()     # Set model to evaluate mode

        running_loss = 0.0
        tot_cnt = 0

        # Iterate over data.
        current_data = dataloader[phase].load_data()
        while not current_data["reset"]:
            input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels, sent_level_EEG = current_data["data"]

            input_embeddings_batch = input_embeddings.to(staging_device, dtype=torch.float16)
            input_masks_batch = input_masks.to(staging_device, dtype=torch.float16)
            input_mask_invert_batch = input_mask_invert.to(staging_device, dtype=torch.float16)
            target_ids_batch = target_ids.to(staging_device)

            """replace padding ids in target_ids with -100"""
            target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

            optimizer.zero_grad()

            args_dict = {
                "input_data_batch" : input_embeddings_batch,
                "input_masks_batch" : input_masks_batch,
                "input_masks_invert" : input_mask_invert_batch,
                "target_ids_batch" : target_ids_batch,
                "pool_result" : False
                }

            seq2seqLMoutput = model(
                mode="EEG-TEXT-BART",
                args_dict=args_dict,
                staging_device=staging_device
                )

            # Use the BART language modeling loss
            loss = seq2seqLMoutput.loss

            # Backward + Optimize only if in training phase
            if phase == 'train':
                if device_ids == None:
                    loss.backward()
                    optimizer.step()
                else:
                    loss.mean().backward()
                    optimizer.step()

            # Compute stats
            if device_ids == None:
                running_loss += loss.item() * input_embeddings_batch.size()[0]
            else:
                running_loss += loss.mean().item() * input_embeddings_batch.size()[0]
            tot_cnt += input_embeddings_batch.size()[0]
            current_data = dataloader[phase].load_data()

        epoch_loss = running_loss / tot_cnt

        results[f"{phase}_loss"] = epoch_loss
    results["model"] = model
    return results

In [22]:
dataloader = dataset_dict["ZuCo-BART"]
model = model
optimizer = optim.Adam(model.parameters(), lr=5e-3)
from transformers import BartTokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
criterion = nn.CrossEntropyLoss()
device = "cuda"
device_ids = None
staging_device = "cuda"

In [23]:
num_epochs = 10

In [24]:
for epoch_num in range(num_epochs):
    args_dict = {
        "dataloader" : dataloader,
        "model" : model,
        "optimizer" : optimizer,
        "tokenizer" : tokenizer,
        "criterion" : criterion,
        "device" : device,
        "device_ids" : device_ids,
        "staging_device" : staging_device
    }
    results = train(args_dict)
    model = results["model"]
    print(f"Epoch {epoch_num} Train Loss: {results['train_loss']} Dev Loss: {results['dev_loss']}")

RuntimeError: Tensors of the same index must be on the same device and the same dtype except `step` tensors that can be CPU and float32 notwithstanding